# LeetCode 3235 — Check if the Rectangle Corner Is Reachable bằng Python

- Link: https://leetcode.com/problems/check-if-the-rectangle-corner-is-reachable/
- Độ khó: **Hard**
- Chủ đề: **Đối ngẫu "đường đi ⇔ không có rào chắn"**: đường từ góc dưới-trái tới góc trên-phải bị chặn ⇔ có một chuỗi hình tròn giao nhau nối **cạnh trái/trên** với **cạnh phải/dưới**; **DSU** trên các hình tròn, cẩn thận điều kiện giao nhau **bên trong** hình chữ nhật

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def canReachCorner(self, xCorner: int, yCorner: int, circles: List[List[int]]) -> bool:
```

## 1. Mục tiêu học

1. Biến bài toán tìm đường liên tục thành bài toán **đồ thị rời rạc**: các hình tròn là đỉnh; hai hình tròn chồng/chạm nhau là cạnh.
2. Lập luận đối ngẫu: đường từ `(0, 0)` tới `(X, Y)` (bên trong hình chữ nhật) **không tồn tại** ⇔ một thành phần liên thông các hình tròn chạm **(cạnh trái hoặc cạnh trên)** và **(cạnh phải hoặc cạnh dưới)** — nó cắt đôi hình chữ nhật thành phần chứa góc đầu và phần chứa góc cuối.
3. Trường hợp riêng: hình tròn **chứa một trong hai góc** ⇒ không thể.
4. Chi tiết tinh tế: hai hình tròn chạm nhau **ngoài** hình chữ nhật không tạo rào chắn bên trong — kiểm tra một điểm của vùng giao nằm trong hình chữ nhật.
5. Kiểm chứng bằng **loang trên lưới mịn** (chỉ dùng cho cấu hình không suy biến).

## 2. Đề bài & ví dụ

Hình chữ nhật góc dưới-trái `(0, 0)`, góc trên-phải `(X, Y)`. Các hình tròn `[x_i, y_i, r_i]`. Hỏi có đường đi từ góc dưới-trái tới góc trên-phải sao cho toàn bộ đường đi nằm **trong** hình chữ nhật, **không chạm hay đi vào** hình tròn nào, và chỉ chạm biên hình chữ nhật tại **hai góc**.

**Ví dụ 1:** `X = 3, Y = 4`, `[[2,1,1]]` → `true`. **Ví dụ 2:** `3, 3`, `[[1,1,2]]` → `false`.
**Ví dụ 3:** `3, 3`, `[[2,1,1],[1,2,1]]` → `false`. **Ví dụ 4:** `4, 4`, `[[5,5,1]]` → `true`.

**Ràng buộc:** `3 ≤ X, Y ≤ 10^9`; `1 ≤ len(circles) ≤ 1000`; `1 ≤ x_i, y_i, r_i ≤ 10^9`.

In [1]:
from typing import List
from collections import deque
import math

examples = [(3, 4, [[2, 1, 1]], True), (3, 3, [[1, 1, 2]], False), (3, 3, [[2, 1, 1], [1, 2, 1]], False), (4, 4, [[5, 5, 1]], True)]
for X, Y, c, exp in examples:
    print((X, Y), c, '-> kỳ vọng', exp)

(3, 4) [[2, 1, 1]] -> kỳ vọng True
(3, 3) [[1, 1, 2]] -> kỳ vọng False
(3, 3) [[2, 1, 1], [1, 2, 1]] -> kỳ vọng False
(4, 4) [[5, 5, 1]] -> kỳ vọng True


## 3. Kiến thức nền

### 3.1. Hình tròn chạm cạnh

Với tâm có tọa độ dương (theo ràng buộc), hình tròn chạm **đoạn** cạnh trái (`x = 0`, `0 ≤ y ≤ Y`) nếu `x_i ≤ r_i` và `0 ≤ y_i ≤ Y`; tương tự các cạnh khác. (Khi tâm nằm ngoài dải tương ứng, việc chạm đoạn cạnh chỉ có thể xảy ra gần góc — trường hợp đó hoặc chứa góc, hoặc không ảnh hưởng; xem kiểm chứng ở mục 12.)

In [2]:
def contains(cx, cy, r, px, py):
    return (cx - px) ** 2 + (cy - py) ** 2 <= r * r

def touches_left_or_top(cx, cy, r, X, Y):
    return (cx <= r and 0 <= cy <= Y) or (abs(cy - Y) <= r and 0 <= cx <= X)

def touches_right_or_bottom(cx, cy, r, X, Y):
    return (abs(cx - X) <= r and 0 <= cy <= Y) or (cy <= r and 0 <= cx <= X)

print(touches_left_or_top(1, 2, 1, 3, 3), touches_right_or_bottom(1, 2, 1, 3, 3), touches_right_or_bottom(2, 1, 1, 3, 3))

True False True


### 3.2. Hai hình tròn giao nhau **trong** hình chữ nhật

Giao nhau (chồng hoặc chạm): `(x₁ − x₂)² + (y₁ − y₂)² ≤ (r₁ + r₂)²`. Điểm `P` chia đoạn nối tâm theo tỉ lệ `r₁ : r₂` (`P = (x₁r₂ + x₂r₁)/(r₁+r₂)`, tương tự `y`) nằm trong vùng giao khi hai hình tròn cắt nhau (không lồng nhau hoàn toàn). Lời giải chuẩn dùng điều kiện `P_x < X` và `P_y < Y` (nhân chéo để tránh số thực) để loại các cặp chỉ giao nhau ở **ngoài** phía trên/phải.

## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Ví dụ 3: hình tròn `(1, 2, 1)` chạm cạnh trái; `(2, 1, 1)` chạm cạnh dưới; hai hình tròn giao nhau ⇒ một "bức tường" nối cạnh trái với cạnh dưới, khép kín góc `(0, 0)` ⇒ `false`.

Ví dụ 4: `(5, 5, 1)` nằm ngoài hình chữ nhật `4 × 4`, không chứa góc `(4, 4)` (khoảng cách `√2 > 1`), không chạm cạnh nào ⇒ `true`.

## 5. Cách 1 — Brute Force: loang trên lưới mịn (oracle gần đúng)

In [3]:
def grid_flood(X, Y, circles, S=12):
    W, H = X * S, Y * S
    def free(i, j):
        x, y = i / S, j / S
        return all((x - a) ** 2 + (y - b) ** 2 > r * r for a, b, r in circles)
    start, goal = (0, 0), (W, H)
    if not free(*start) or not free(*goal):
        return False
    seen = {start}; q = deque([start])
    while q:
        i, j = q.popleft()
        if (i, j) == goal:
            return True
        for di in (-1, 0, 1):
            for dj in (-1, 0, 1):
                a, b = i + di, j + dj
                if (a, b) in seen or (di == 0 and dj == 0):
                    continue
                if ((a, b) == goal or (0 < a < W and 0 < b < H)) and free(a, b):
                    seen.add((a, b)); q.append((a, b))
    return False

for X, Y, c, exp in examples:
    print(grid_flood(X, Y, c), '| kỳ vọng', exp)

True | kỳ vọng True
False | kỳ vọng False
False | kỳ vọng False
True | kỳ vọng True


Đường đi chỉ qua các điểm lưới `(i/S, j/S)` nằm **trong** hình chữ nhật (trừ hai góc) và ngoài mọi hình tròn. Chỉ đáng tin khi cấu hình **không suy biến** (không có tiếp xúc "sát nút" giữa các hình tròn hay với cạnh) — ta lọc các ca đó khi sinh test ngẫu nhiên.

## 6. Ý tưởng thuật toán / bước đột phá

```
nếu hình tròn nào chứa (0, 0) hoặc (X, Y): return False
DSU với hai nút ảo LT (trái/trên) và RB (phải/dưới)
mỗi hình tròn: chạm trái/trên -> nối LT; chạm phải/dưới -> nối RB
mỗi cặp hình tròn giao nhau (và điểm giao "trong" hình chữ nhật) -> nối
return find(LT) != find(RB)
```

`O(n² α(n))` với `n ≤ 1000`.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — DFS từ các hình tròn chạm trái/trên

In [4]:
def dfs_solution(X, Y, circles):
    n = len(circles)
    for x, y, r in circles:
        if contains(x, y, r, 0, 0) or contains(x, y, r, X, Y):
            return False
    def linked(i, j):
        x1, y1, r1 = circles[i]; x2, y2, r2 = circles[j]
        return (x1 - x2) ** 2 + (y1 - y2) ** 2 <= (r1 + r2) ** 2 and \
            x1 * r2 + x2 * r1 < (r1 + r2) * X and y1 * r2 + y2 * r1 < (r1 + r2) * Y
    seen = [False] * n
    for s in range(n):
        if seen[s] or not touches_left_or_top(*circles[s], X, Y):
            continue
        seen[s] = True; st = [s]
        while st:
            u = st.pop()
            if touches_right_or_bottom(*circles[u], X, Y):
                return False
            for v in range(n):
                if not seen[v] and linked(u, v):
                    seen[v] = True; st.append(v)
    return True

for X, Y, c, exp in examples:
    print(dfs_solution(X, Y, c), '| kỳ vọng', exp)

True | kỳ vọng True
False | kỳ vọng False
False | kỳ vọng False
True | kỳ vọng True


### 7.2. Cách 3 — DSU với hai nút ảo (lời giải chính)

In [5]:
def dsu_solution(X, Y, circles):
    n = len(circles)
    par = list(range(n + 2))
    def find(a):
        while par[a] != a:
            par[a] = par[par[a]]; a = par[a]
        return a
    def union(a, b):
        par[find(a)] = find(b)
    LT, RB = n, n + 1
    for i, (x, y, r) in enumerate(circles):
        if contains(x, y, r, 0, 0) or contains(x, y, r, X, Y):
            return False
        if touches_left_or_top(x, y, r, X, Y):
            union(i, LT)
        if touches_right_or_bottom(x, y, r, X, Y):
            union(i, RB)
        for j in range(i):
            a, b, s = circles[j]
            if (x - a) ** 2 + (y - b) ** 2 <= (r + s) ** 2 and x * s + a * r < (r + s) * X and y * s + b * r < (r + s) * Y:
                union(i, j)
    return find(LT) != find(RB)

for X, Y, c, exp in examples:
    print(dsu_solution(X, Y, c), '| kỳ vọng', exp)

True | kỳ vọng True
False | kỳ vọng False
False | kỳ vọng False
True | kỳ vọng True


### 7.3. Cách 4 — BFS đa nguồn từ "bức tường" trái/trên

Đưa mọi hình tròn chạm trái/trên vào hàng đợi cùng lúc, loang theo quan hệ giao nhau; nếu chạm một hình tròn thuộc phải/dưới thì bị chặn. Tương đương cách 2 nhưng nhìn như "loang từ một bức tường".

In [6]:
def multi_source_bfs(X, Y, circles):
    n = len(circles)
    if any(contains(x, y, r, 0, 0) or contains(x, y, r, X, Y) for x, y, r in circles):
        return False
    q = deque(i for i in range(n) if touches_left_or_top(*circles[i], X, Y))
    seen = set(q)
    while q:
        u = q.popleft()
        if touches_right_or_bottom(*circles[u], X, Y):
            return False
        x1, y1, r1 = circles[u]
        for v in range(n):
            if v in seen:
                continue
            x2, y2, r2 = circles[v]
            if (x1 - x2) ** 2 + (y1 - y2) ** 2 <= (r1 + r2) ** 2 and \
               x1 * r2 + x2 * r1 < (r1 + r2) * X and y1 * r2 + y2 * r1 < (r1 + r2) * Y:
                seen.add(v); q.append(v)
    return True

for X, Y, c, exp in examples:
    print(multi_source_bfs(X, Y, c), '| kỳ vọng', exp)

True | kỳ vọng True
False | kỳ vọng False
False | kỳ vọng False
True | kỳ vọng True


## 8. TRACE từng bước

Trace cách 3 trên ví dụ 3 và trên một cấu hình "tường" ba hình tròn.

In [7]:
def trace(X, Y, circles):
    print(f'hình chữ nhật {X}×{Y}')
    for i, (x, y, r) in enumerate(circles):
        print(f'  C{i} = {(x, y, r)}: chứa góc đầu/cuối? {contains(x, y, r, 0, 0)}/{contains(x, y, r, X, Y)}; '
              f'chạm trái/trên {touches_left_or_top(x, y, r, X, Y)}, phải/dưới {touches_right_or_bottom(x, y, r, X, Y)}')
    for i in range(len(circles)):
        for j in range(i):
            (x, y, r), (a, b, s) = circles[i], circles[j]
            inter = (x - a) ** 2 + (y - b) ** 2 <= (r + s) ** 2
            inside = x * s + a * r < (r + s) * X and y * s + b * r < (r + s) * Y
            if inter:
                print(f'  C{j} – C{i}: giao nhau, điểm giao trong hình chữ nhật: {inside}')
    print('đáp án:', dsu_solution(X, Y, circles), '| loang lưới:', grid_flood(X, Y, circles))

trace(*examples[2][:3])
trace(10, 6, [[1, 5, 2], [4, 3, 2], [7, 2, 2]])

hình chữ nhật 3×3
  C0 = (2, 1, 1): chứa góc đầu/cuối? False/False; chạm trái/trên False, phải/dưới True
  C1 = (1, 2, 1): chứa góc đầu/cuối? False/False; chạm trái/trên True, phải/dưới False
  C0 – C1: giao nhau, điểm giao trong hình chữ nhật: True
đáp án: False | loang lưới: False
hình chữ nhật 10×6
  C0 = (1, 5, 2): chứa góc đầu/cuối? False/False; chạm trái/trên True, phải/dưới False
  C1 = (4, 3, 2): chứa góc đầu/cuối? False/False; chạm trái/trên False, phải/dưới False
  C2 = (7, 2, 2): chứa góc đầu/cuối? False/False; chạm trái/trên False, phải/dưới True
  C0 – C1: giao nhau, điểm giao trong hình chữ nhật: True
  C1 – C2: giao nhau, điểm giao trong hình chữ nhật: True
đáp án: False | loang lưới: False


## 9. Code nộp LeetCode

In [8]:
class Solution:
    def canReachCorner(self, xCorner: int, yCorner: int, circles: List[List[int]]) -> bool:
        X, Y = xCorner, yCorner
        n = len(circles)
        par = list(range(n + 2))

        def find(a):
            while par[a] != a:
                par[a] = par[par[a]]
                a = par[a]
            return a

        LT, RB = n, n + 1
        for i, (x, y, r) in enumerate(circles):
            rr = r * r
            if x * x + y * y <= rr or (x - X) ** 2 + (y - Y) ** 2 <= rr:
                return False
            if (x <= r and y <= Y) or (abs(y - Y) <= r and x <= X):
                par[find(i)] = find(LT)
            if (abs(x - X) <= r and y <= Y) or (y <= r and x <= X):
                par[find(i)] = find(RB)
            for j in range(i):
                a, b, s = circles[j]
                if (x - a) ** 2 + (y - b) ** 2 <= (r + s) ** 2 and x * s + a * r < (r + s) * X and y * s + b * r < (r + s) * Y:
                    par[find(i)] = find(j)
        return find(LT) != find(RB)

for X, Y, c, exp in examples:
    got = Solution().canReachCorner(X, Y, c)
    print(got, 'OK' if got == exp else 'SAI')

True OK
False OK
False OK
True OK


Do `x, y ≥ 1` (ràng buộc), các điều kiện `0 ≤ y` và `0 ≤ x` luôn đúng nên được lược bỏ.

## 10. Vì sao thuật toán đúng

1. **Rào chắn ⇒ không có đường:** một thành phần liên thông `K` (hợp các hình tròn đóng) chạm cạnh trái/trên và cạnh phải/dưới tạo thành một tập liên thông nối hai "nửa biên" — mọi đường từ `(0, 0)` tới `(X, Y)` trong hình chữ nhật phải cắt `K` (định lý đường cong Jordan).
2. **Không có rào chắn ⇒ có đường:** đi dọc theo "biên" của hợp các thành phần chạm trái/trên — vì không thành phần nào chạm cả hai phía, vùng tự do chứa một đường nối hai góc (lập luận đối ngẫu chuẩn cho bài "lối thoát giữa các chướng ngại").
3. **Góc bị chứa:** nếu `(0, 0)` hoặc `(X, Y)` nằm trong/trên một hình tròn, đường đi không thể bắt đầu/kết thúc.
4. **Giao nhau trong hình chữ nhật:** chỉ những chỗ giao nằm trong hình chữ nhật mới tạo rào chắn bên trong; điều kiện điểm chia `P` loại bỏ các cặp giao nhau ở ngoài phía phải/trên (với tâm dương, giao nhau ở phía trái/dưới ngoài hình chữ nhật đồng nghĩa với việc cả hai đều chạm cạnh tương ứng nên không làm thay đổi kết luận).

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Loang lưới | `O(X Y S² n)` | `O(X Y S²)` |
| 2. DFS trên hình tròn | `O(n²)` | `O(n)` |
| 3. DSU | `O(n² α(n))` | `O(n)` |
| 4. BFS đa nguồn | `O(n²)` | `O(n)` |

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [9]:
impls = {'dfs': dfs_solution, 'dsu': dsu_solution, 'bfs': multi_source_bfs, 'submit': lambda X, Y, c: Solution().canReachCorner(X, Y, c)}
edge = [
    (4, 4, [[6, 6, 3]], False),                     # chứa góc (4, 4) từ ngoài
    (30, 30, [[10, 70, 43], [70, 10, 43]], True),   # giao nhau ngoài hình chữ nhật
    (3, 3, [[1, 1, 1]], False),                     # tiếp xúc cạnh trái và cạnh dưới
    (10, 6, [[1, 5, 2], [4, 3, 2], [7, 2, 2]], False),
    (10, 10, [[5, 5, 1]], True),
]
for X, Y, c, exp in examples + edge:
    for name, f in impls.items():
        assert f(X, Y, c) == exp, (name, X, Y, c)
print('Ví dụ đề và ca biên: tất cả đúng.')
print('đối chiếu lưới cho các ca biên:', [grid_flood(X, Y, c, S=4) for X, Y, c, _ in edge])

Ví dụ đề và ca biên: tất cả đúng.
đối chiếu lưới cho các ca biên: [False, True, False, False, True]


### 12.2. Đối chiếu ngẫu nhiên với loang lưới (loại cấu hình suy biến)

In [10]:
import random
rng = random.Random(3235)
def degenerate(X, Y, circles, eps=0.35):
    for i, (x, y, r) in enumerate(circles):
        for d in [x, y, abs(x - X), abs(y - Y), math.hypot(x, y), math.hypot(x - X, y - Y), math.hypot(x, y - Y), math.hypot(x - X, y)]:
            if abs(d - r) < eps:
                return True
        for j in range(i):
            a, b, s = circles[j]; d = math.hypot(x - a, y - b)
            if abs(d - (r + s)) < eps or abs(d - abs(r - s)) < eps:
                return True
    return False

cnt = 0
while cnt < 250:
    X, Y = rng.randint(3, 7), rng.randint(3, 7)
    cs = [[rng.randint(1, 9), rng.randint(1, 9), rng.randint(1, 3)] for _ in range(rng.randint(1, 4))]
    if degenerate(X, Y, cs):
        continue
    exp = grid_flood(X, Y, cs)
    for name, f in impls.items():
        assert f(X, Y, cs) == exp, (name, X, Y, cs)
    cnt += 1
print('Khớp loang lưới trên', cnt, 'cấu hình không suy biến.')

Khớp loang lưới trên 250 cấu hình không suy biến.


### 12.3. Ca lớn: 1000 hình tròn, tọa độ tới `10^9`

In [11]:
import time
big = {
    'ngẫu nhiên': (10 ** 9, 10 ** 9, [[rng.randint(1, 10 ** 9), rng.randint(1, 10 ** 9), rng.randint(1, 10 ** 7)] for _ in range(1000)]),
    'chuỗi chéo chặn': (10 ** 9, 10 ** 9, [[10 ** 6 * (k + 1), 10 ** 9 - 10 ** 6 * (k + 1), 10 ** 6] for k in range(999)] + [[10 ** 9, 10 ** 6, 2 * 10 ** 6]]),
}
for name, (X, Y, c) in big.items():
    t = time.time(); a = Solution().canReachCorner(X, Y, c); t1 = time.time() - t
    t = time.time(); b = multi_source_bfs(X, Y, c); t2 = time.time() - t
    assert a == b
    print(f"{name:16s}: {a} | nộp {t1:.2f}s | BFS {t2:.2f}s")

ngẫu nhiên      : True | nộp 0.17s | BFS 0.01s


chuỗi chéo chặn : False | nộp 0.15s | BFS 0.12s


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Quên kiểm tra hình tròn chứa góc

In [12]:
def wrong_no_corner(X, Y, circles):
    n = len(circles); par = list(range(n + 2))
    def find(a):
        while par[a] != a:
            a = par[a]
        return a
    for i, (x, y, r) in enumerate(circles):
        if touches_left_or_top(x, y, r, X, Y):
            par[find(i)] = find(n)
        if touches_right_or_bottom(x, y, r, X, Y):
            par[find(i)] = find(n + 1)
        for j in range(i):
            a, b, s = circles[j]
            if (x - a) ** 2 + (y - b) ** 2 <= (r + s) ** 2 and x * s + a * r < (r + s) * X and y * s + b * r < (r + s) * Y:
                par[find(i)] = find(j)
    return find(n) != find(n + 1)

print('sai :', wrong_no_corner(4, 4, [[6, 6, 3]]), '| đúng:', Solution().canReachCorner(4, 4, [[6, 6, 3]]))

sai : True | đúng: False


**Vì sao sai:** tâm `(6, 6)` nằm ngoài, không thỏa điều kiện "chạm cạnh" theo dải, nhưng hình tròn bán kính 3 chứa góc `(4, 4)` (khoảng cách `√8 < 3`).

### Lỗi 2 — Nối hai hình tròn chỉ vì chúng giao nhau (dù giao ở ngoài)

In [13]:
def wrong_any_intersection(X, Y, circles):
    n = len(circles); par = list(range(n + 2))
    def find(a):
        while par[a] != a:
            a = par[a]
        return a
    for i, (x, y, r) in enumerate(circles):
        if contains(x, y, r, 0, 0) or contains(x, y, r, X, Y):
            return False
        if touches_left_or_top(x, y, r, X, Y):
            par[find(i)] = find(n)
        if touches_right_or_bottom(x, y, r, X, Y):
            par[find(i)] = find(n + 1)
        for j in range(i):
            a, b, s = circles[j]
            if (x - a) ** 2 + (y - b) ** 2 <= (r + s) ** 2:          # SAI: thiếu điều kiện "trong hình chữ nhật"
                par[find(i)] = find(j)
    return find(n) != find(n + 1)

bad = (30, 30, [[10, 70, 43], [70, 10, 43]])
print('sai :', wrong_any_intersection(*bad), '| đúng:', Solution().canReachCorner(*bad), '| lưới:', grid_flood(*bad, S=3))

sai : False | đúng: True | lưới: True


**Vì sao sai:** hình tròn trên chạm cạnh trên, hình tròn phải chạm cạnh phải; chúng giao nhau quanh `(40, 40)` — **ngoài** hình chữ nhật `30 × 30` — nên vẫn còn khe hở gần góc `(30, 30)`.

### Lỗi 3 — Dùng `<` thay vì `≤` khi xét chạm cạnh (bỏ qua tiếp xúc)

In [14]:
def wrong_strict_touch(X, Y, circles):
    n = len(circles); par = list(range(n + 2))
    def find(a):
        while par[a] != a:
            a = par[a]
        return a
    for i, (x, y, r) in enumerate(circles):
        if contains(x, y, r, 0, 0) or contains(x, y, r, X, Y):
            return False
        if (x < r and y <= Y) or (abs(y - Y) < r and x <= X):        # SAI: tiếp xúc cũng là "chạm"
            par[find(i)] = find(n)
        if (abs(x - X) < r and y <= Y) or (y < r and x <= X):
            par[find(i)] = find(n + 1)
    return find(n) != find(n + 1)

print('sai :', wrong_strict_touch(3, 3, [[1, 1, 1]]), '| đúng:', Solution().canReachCorner(3, 3, [[1, 1, 1]]))

sai : True | đúng: False


**Vì sao sai:** hình tròn `(1, 1, 1)` tiếp xúc cạnh trái tại `(0, 1)` và cạnh dưới tại `(1, 0)`: đường đi không được chạm hình tròn **và** không được chạm biên (trừ góc) ⇒ góc `(0, 0)` bị khép kín.

### Lỗi 4 — Ghép nhầm nhóm cạnh (trái + dưới) và (phải + trên)

In [15]:
def wrong_grouping(X, Y, circles):
    n = len(circles); par = list(range(n + 2))
    def find(a):
        while par[a] != a:
            a = par[a]
        return a
    for i, (x, y, r) in enumerate(circles):
        if contains(x, y, r, 0, 0) or contains(x, y, r, X, Y):
            return False
        if (x <= r and y <= Y) or (y <= r and x <= X):                # SAI: trái + dưới
            par[find(i)] = find(n)
        if (abs(x - X) <= r and y <= Y) or (abs(y - Y) <= r and x <= X):   # SAI: phải + trên
            par[find(i)] = find(n + 1)
        for j in range(i):
            a, b, s = circles[j]
            if (x - a) ** 2 + (y - b) ** 2 <= (r + s) ** 2 and x * s + a * r < (r + s) * X and y * s + b * r < (r + s) * Y:
                par[find(i)] = find(j)
    return find(n) != find(n + 1)

print('sai :', wrong_grouping(3, 3, [[1, 1, 1]]), '| đúng:', Solution().canReachCorner(3, 3, [[1, 1, 1]]))

sai : True | đúng: False


**Vì sao sai:** hình tròn `(1, 1, 1)` chạm cạnh trái và cạnh dưới — một bức tường nối **trái với dưới** khép kín góc đầu. Bản sai xếp hai cạnh này vào **cùng** nhóm nên không thấy rào chắn. Phải ghép trái với **trên** (một phía của đường đi) và phải với **dưới** (phía kia).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Có đường đi tránh chướng ngại trong vùng?" ⇒ đối ngẫu: "có chuỗi chướng ngại nối hai phần biên?" ⇒ DSU/BFS trên chướng ngại.
- Cạnh biên chia thành **hai nửa** theo hai điểm đầu/cuối (ở đây: trái+trên vs phải+dưới).
- Hình học số nguyên lớn: so sánh bình phương, nhân chéo — tránh số thực.

## 15. Biến thể của bài

1. Đường đi giữa hai cạnh đối diện (trái → phải) ⇒ rào chắn nối **trên** với **dưới**.
2. Chướng ngại là hình chữ nhật/đoạn thẳng ⇒ thay phép kiểm tra giao.
3. LC 2852-ish "escape the spreading fire"…; LC 1036 (Escape a Large Maze) — cùng ý tưởng rào chắn.
4. Tìm bán kính lớn nhất của "quả bóng" đi qua được ⇒ nhị phân + kiểm tra (phóng to mọi hình tròn thêm `ρ`).

In [16]:
# Biến thể 1: từ cạnh trái sang cạnh phải (đi trong dải), bị chặn nếu chuỗi hình tròn nối cạnh trên với cạnh dưới
def left_to_right(X, Y, circles):
    n = len(circles); par = list(range(n + 2))
    def find(a):
        while par[a] != a:
            a = par[a]
        return a
    for i, (x, y, r) in enumerate(circles):
        if abs(y - Y) <= r:
            par[find(i)] = find(n)
        if y <= r:
            par[find(i)] = find(n + 1)
        for j in range(i):
            a, b, s = circles[j]
            if (x - a) ** 2 + (y - b) ** 2 <= (r + s) ** 2:
                par[find(i)] = find(j)
    return find(n) != find(n + 1)

def strip_grid(X, Y, circles, S=8):
    W, H = X * S, Y * S
    free = lambda i, j: all((i / S - a) ** 2 + (j / S - b) ** 2 > r * r for a, b, r in circles)
    q = deque((0, j) for j in range(1, H) if free(0, j)); seen = set(q)
    while q:
        i, j = q.popleft()
        if i == W:
            return True
        for di in (-1, 0, 1):
            for dj in (-1, 0, 1):
                a, b = i + di, j + dj
                if 0 <= a <= W and 0 < b < H and (a, b) not in seen and free(a, b):
                    seen.add((a, b)); q.append((a, b))
    return False

rng = random.Random(15); cnt = 0
while cnt < 100:
    X, Y = rng.randint(6, 10), rng.randint(3, 6)
    cs = [[rng.randint(1, X - 1), rng.randint(0, Y), rng.randint(1, 2)] for _ in range(rng.randint(1, 4))]
    if degenerate(X, Y, cs) or any(x - r <= 0.4 or x + r >= X - 0.4 for x, y, r in cs):
        continue
    assert left_to_right(X, Y, cs) == strip_grid(X, Y, cs)
    cnt += 1
print('Biến thể "trái sang phải" (rào chắn trên–dưới) khớp loang lưới trên', cnt, 'cấu hình.')

Biến thể "trái sang phải" (rào chắn trên–dưới) khớp loang lưới trên 100 cấu hình.


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Ghi chú |
|---|---|---|---|
| 1 | Loang lưới mịn | lớn | Oracle (không suy biến) |
| 2 | DFS từ tường trái/trên | `O(n²)` | Trực quan |
| 3 | DSU hai nút ảo | `O(n² α)` | **Nộp bài** |
| 4 | BFS đa nguồn | `O(n²)` | Cùng ý tưởng |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `X = 5, Y = 5`, `[[3, 3, 1]]`.

**Bài 2 (tính tay):** `X = 6, Y = 6`, `[[1, 4, 2], [4, 1, 2]]`.

**Bài 3 (điền TODO):** trả về luôn **thành phần** hình tròn tạo rào chắn (danh sách chỉ số) nếu bị chặn.

**Bài 4 (tự viết lại):** biến thể 4 — bán kính lớn nhất `ρ` của một đĩa đi được từ góc này sang góc kia (đơn giản hóa: phóng to các hình tròn thêm `ρ`, nhị phân với độ chính xác `10^{-3}`, chỉ kiểm tra rào chắn giữa các hình tròn, bỏ qua biên).

**Bài 5 (mini challenge):** đếm số hình tròn có thể **xóa đi một cái** để đường đi tồn tại (n nhỏ).

In [17]:
def barrier_component_todo(X, Y, circles):
    # TODO
    raise NotImplementedError

def max_disk_todo(X, Y, circles):
    # TODO
    raise NotImplementedError

def removable_count_todo(X, Y, circles):
    # TODO
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Không chạm cạnh, không chứa góc. (2) Không rào chắn. (3) `true`.

**Bài 2.** (1) `(1, 4, 2)` chạm trái; `(4, 1, 2)` chạm dưới. (2) Khoảng cách tâm `√18 ≈ 4.24 ≤ 4` ? Không ⇒ không giao. (3) `true`.

**Bài 3.** (1) Sau DSU, lấy các `i` cùng gốc với `LT` khi `find(LT) == find(RB)`. (2) Trả danh sách. (3) Kiểm thành phần đó chạm cả hai phía.

**Bài 4.** (1) Đĩa bán kính `ρ` đi qua ⇔ tâm đĩa đi qua khi mọi hình tròn phóng to thêm `ρ`. (2) Tính chất đơn điệu ⇒ nhị phân. (3) Biên cũng "phóng to" — bản đơn giản bỏ qua.

**Bài 5.** (1) Thử xóa từng hình tròn. (2) Gọi lại lời giải. (3) `O(n³)` cho `n` nhỏ.

## 19. Đáp án tham khảo cho bài tập

In [18]:
# Bài 1, 2, 3
assert Solution().canReachCorner(5, 5, [[3, 3, 1]]) is True
assert Solution().canReachCorner(6, 6, [[1, 4, 2], [4, 1, 2]]) is True

def barrier_component(X, Y, circles):
    n = len(circles); par = list(range(n + 2))
    def find(a):
        while par[a] != a:
            a = par[a]
        return a
    for i, (x, y, r) in enumerate(circles):
        if touches_left_or_top(x, y, r, X, Y):
            par[find(i)] = find(n)
        if touches_right_or_bottom(x, y, r, X, Y):
            par[find(i)] = find(n + 1)
        for j in range(i):
            a, b, s = circles[j]
            if (x - a) ** 2 + (y - b) ** 2 <= (r + s) ** 2 and x * s + a * r < (r + s) * X and y * s + b * r < (r + s) * Y:
                par[find(i)] = find(j)
    if find(n) != find(n + 1):
        return []
    return [i for i in range(n) if find(i) == find(n)]

print('Bài 1: true | Bài 2: true | Bài 3 (ví dụ 3):', barrier_component(*examples[2][:3]),
      '| (tường 3 hình tròn):', barrier_component(10, 6, [[1, 5, 2], [4, 3, 2], [7, 2, 2]]))

Bài 1: true | Bài 2: true | Bài 3 (ví dụ 3): [0, 1] | (tường 3 hình tròn): [0, 1, 2]


In [19]:
# Bài 4, 5
def max_disk(X, Y, circles, iters=40):
    def blocked(rho):
        n = len(circles); par = list(range(n))
        def find(a):
            while par[a] != a:
                a = par[a]
            return a
        for i in range(n):
            for j in range(i):
                (x, y, r), (a, b, s) = circles[i], circles[j]
                if math.hypot(x - a, y - b) <= r + s + 2 * rho:
                    par[find(i)] = find(j)
        groups = {}
        for i, (x, y, r) in enumerate(circles):
            R = r + rho
            g = groups.setdefault(find(i), [False, False])
            g[0] |= x <= R or abs(y - Y) <= R
            g[1] |= abs(x - X) <= R or y <= R
        return any(a and b for a, b in groups.values())
    lo, hi = 0.0, min(X, Y) / 2
    for _ in range(iters):
        mid = (lo + hi) / 2
        lo, hi = (lo, mid) if blocked(mid) else (mid, hi)
    return lo

def removable_count(X, Y, circles):
    return sum(1 for i in range(len(circles)) if Solution().canReachCorner(X, Y, circles[:i] + circles[i + 1:]))

print('Bài 4: bán kính đĩa lớn nhất (đơn giản hóa) với 1 hình tròn giữa 10×10:', round(max_disk(10, 10, [[5, 5, 1]]), 3))
print('Bài 5 (tường 3 hình tròn): xóa được', removable_count(10, 6, [[1, 5, 2], [4, 3, 2], [7, 2, 2]]), 'cách')

Bài 4: bán kính đĩa lớn nhất (đơn giản hóa) với 1 hình tròn giữa 10×10: 4.0
Bài 5 (tường 3 hình tròn): xóa được 3 cách


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
nếu hình tròn chứa (0,0) hoặc (X,Y): False
LT nếu (x ≤ r và y ≤ Y) hoặc (|y−Y| ≤ r và x ≤ X);   RB nếu (|x−X| ≤ r và y ≤ Y) hoặc (y ≤ r và x ≤ X)
nối i, j nếu d² ≤ (r_i+r_j)² và x_i r_j + x_j r_i < (r_i+r_j) X và y_i r_j + y_j r_i < (r_i+r_j) Y
return find(LT) != find(RB)
```

**Checklist:**

- [ ] Kiểm tra hình tròn chứa góc (kể cả tâm ngoài hình chữ nhật)?
- [ ] Nhóm cạnh: **trái + trên** đấu với **phải + dưới**?
- [ ] Tiếp xúc tính là chạm (`≤`)?
- [ ] Chỉ nối hai hình tròn khi chỗ giao nằm trong hình chữ nhật?
- [ ] Số học số nguyên (bình phương, nhân chéo)?